In [ ]:
pip install datasets

In [ ]:
pip install scikit-learn

In [ ]:
pip install torch

In [ ]:
from datasets import load_dataset
dataset = load_dataset("DeepPavlov/snips")
print(dataset)

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame(dataset['train'])
print(df['label'].unique())
print(df['label'].value_counts())

In [ ]:
print(dataset['train'].features)
print()
# See a few examples
for i in range(5):
    print(dataset['train'][i])

In [ ]:
label_map = {
    0: "AddToPlaylist",
    1: "BookRestaurant", 
    2: "GetWeather",
    3: "PlayMusic",
    4: "RateBook",
    5: "SearchCreativeWork",
    6: "SearchScreeningEvent"
}

# Verify distribution
for k, v in label_map.items():
    count = len(df[df['label'] == k])
    print(f"{v}: {count}")

In [ ]:
from collections import Counter

In [ ]:
# Tokenize all utterances
def tokenize(sentence):
    return sentence.lower().split()
    
# Build vocabulary
counter = Counter()
for sample in dataset['train']:
    counter.update(tokenize(sample['utterance']))

# Create word to index mapping
# 0 = padding, 1 = unknown
word2idx = {'<PAD>': 0, '<UNK>': 1}
for word, freq in counter.items():
    if freq >= 2:  # ignore words appearing only once
        word2idx[word] = len(word2idx)

print(f"Vocabulary size: {len(word2idx)}")

# Dataset Splitting

In [ ]:
from sklearn.model_selection import train_test_split

The dataset already comes pre-split into train and test from HuggingFace.  
However, we do need a validation set, therefore, we'll split dataset['train'] to get validation set(only 10%).

In [ ]:
train_data, val_data = train_test_split(
    dataset['train'].to_pandas(), 
    test_size=0.1,      # 10% for validation
    random_state=100,
    stratify=dataset['train']['label']  # keeps class balance
)
print(f"Train: {len(train_data)}")
print(f"Val: {len(val_data)}")
print(f"Test: {len(dataset['test'])}")

# Dataset Encoder

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
MAX_LEN = 50

def encode(sentence):
    tokens = tokenize(sentence)
    ids = [word2idx.get(t, 1) for t in tokens]  # 1 = <UNK>
    # pad or truncate to MAX_LEN
    ids = ids[:MAX_LEN] + [0] * (MAX_LEN - len(ids))
    return ids

class SNIPSDataset(Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx] if hasattr(self.data, 'iloc') else self.data[idx]
        x = torch.tensor(encode(row['utterance']), dtype=torch.long)
        y = torch.tensor(row['label'], dtype=torch.long)
        return x, y

train_dataset = SNIPSDataset(train_data)
val_dataset = SNIPSDataset(val_data)
test_dataset = SNIPSDataset(dataset['test'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")